# 📊 Notebook 1: EDA & Feature Engineering

> **Titanic - Machine Learning from Disaster**

This notebook covers:
1. Data overview (info, describe, missing values)
2. Univariate analysis (distributions of Age, Fare, Pclass, Sex, Survived)
3. Bivariate analysis (correlations, survival rate by feature)
4. Feature engineering walkthrough
5. Visualizations (seaborn/matplotlib)

---

In [ ]:
# Cell 1: Imports & Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print("✅ Libraries imported successfully")

In [ ]:
# Cell 2: Load Data
import sys
sys.path.insert(0, '..')

from src.data_loader import load_train_test

DATA_DIR = '../data'

train_df, test_df = load_train_test(DATA_DIR, backend='pandas')

print(f"📦 Train shape: {train_df.shape}")
print(f"📦 Test shape:  {test_df.shape}")
print(f"\nTrain columns: {list(train_df.columns)}")

In [ ]:
# Cell 3: Data Overview — info()
print("=" * 60)
print("TRAINING DATA INFO")
print("=" * 60)
train_df.info()

In [ ]:
# Cell 4: Data Overview — describe()
print("\n" + "=" * 60)
print("STATISTICAL SUMMARY (Numeric Columns)")
print("=" * 60)
train_df.describe()

In [ ]:
# Cell 5: Missing Values Analysis
print("\n" + "=" * 60)
print("MISSING VALUES ANALYSIS")
print("=" * 60)

missing_train = train_df.isnull().sum()
missing_pct = (missing_train / len(train_df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing_train,
    'Missing %': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(
    'Missing %', ascending=False
)
print(missing_df)
print(f"\nTotal missing values in train: {missing_train.sum()}")

# Visualize missing values
fig, ax = plt.subplots(figsize=(10, 4))
if not missing_df.empty:
    sns.barplot(x=missing_df.index, y='Missing %', data=missing_df.reset_index(), 
                palette='Reds_r', ax=ax)
    ax.set_title('Missing Values by Feature (%)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Missing Percentage (%)')
    ax.set_xlabel('Feature')
    for i, v in enumerate(missing_df['Missing %']):
        ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Univariate Analysis — Target Variable (Survived)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
survival_counts = train_df['Survived'].value_counts()
colors = ['#e74c3c', '#2ecc71']
labels = ['Did Not Survive (0)', 'Survived (1)']
axes[0].pie(survival_counts, labels=labels, autopct='%1.1f%%', 
               colors=colors, explode=[0.05, 0], startangle=90,
               textprops={'fontsize': 12})
axes[0].set_title('Survival Distribution', fontsize=14, fontweight='bold')

# Bar chart
sns.countplot(x='Survived', data=train_df, palette=colors, ax=axes[1])
axes[1].set_title('Survival Count', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Survived (0 = No, 1 = Yes)')
axes[1].set_ylabel('Passenger Count')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}', 
                    (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=12)

plt.suptitle('Target Variable Analysis: Survived', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Univariate Analysis — Age Distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
sns.histplot(train_df['Age'].dropna(), bins=30, kde=True, color='#3498db', ax=axes[0])
axes[0].set_title('Age Distribution (Histogram)', fontsize=13, fontweight='bold')
axes[0].axvline(train_df['Age'].mean(), color='red', linestyle='--', 
              label=f'Mean: {train_df["Age"].mean():.1f}')
axes[0].axvline(train_df['Age'].median(), color='green', linestyle='--', 
              label=f'Median: {train_df["Age"].median():.1f}')
axes[0].legend()

# Box plot by Survival
sns.boxplot(x='Survived', y='Age', data=train_df, palette=['#e74c3c', '#2ecc71'], ax=axes[1])
axes[1].set_title('Age by Survival Status', fontsize=13, fontweight='bold')

# Violin plot by Sex
sns.violinplot(x='Sex', y='Age', hue='Survived', split=True, 
                data=train_df, palette=['#e74c3c', '#2ecc71'], ax=axes[2])
axes[2].set_title('Age by Sex & Survival', fontsize=13, fontweight='bold')

plt.suptitle('Univariate Analysis: Age', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Univariate Analysis — Fare Distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram (with outliers visible)
sns.histplot(train_df['Fare'].dropna(), bins=40, kde=True, color='#9b59b6', ax=axes[0])
axes[0].set_title('Fare Distribution (All)', fontsize=13, fontweight='bold')

# Histogram (clipped for better view)
fare_clipped = train_df[train_df['Fare'] < 200]['Fare']
sns.histplot(fare_clipped, bins=30, kde=True, color='#9b59b6', ax=axes[1])
axes[1].set_title('Fare Distribution (< $200)', fontsize=13, fontweight='bold')

# Box plot by Pclass
sns.boxplot(x='Pclass', y='Fare', data=train_df[train_df['Fare'] < 200], 
            palette='Blues_d', ax=axes[2])
axes[2].set_title('Fare by Passenger Class', fontsize=13, fontweight='bold')

plt.suptitle('Univariate Analysis: Fare', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Univariate Analysis — Pclass, Sex, Embarked
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pclass
pclass_counts = train_df['Pclass'].value_counts().sort_index()
colors_pclass = ['#3498db', '#e74c3c', '#2ecc71']
axes[0].bar(pclass_counts.index.astype(str), pclass_counts.values, color=colors_pclass)
axes[0].set_title('Passenger Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Pclass')
axes[0].set_ylabel('Count')
for i, v in enumerate(pclass_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=11)

# Sex
sex_counts = train_df['Sex'].value_counts()
axes[1].pie(sex_counts, labels=sex_counts.index, autopct='%1.1f%%', 
           colors=['#3498db', '#e74c3c'], startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Gender Distribution', fontsize=13, fontweight='bold')

# Embarked
embarked_counts = train_df['Embarked'].value_counts()
axes[2].bar(embarked_counts.index, embarked_counts.values, color=['#f39c12', '#1abc9c', '#9b59b6'])
axes[2].set_title('Port of Embarkation', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Port (S=Southampton, C=Cherbourg, Q=Queenstown)')
axes[2].set_ylabel('Count')
for i, v in enumerate(embarked_counts.values):
    axes[2].text(i, v + 10, str(v), ha='center', fontsize=11)

plt.suptitle('Categorical Features: Pclass, Sex, Embarked', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Bivariate Analysis — Correlation Heatmap
# Select numeric columns only
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
corr_matrix = train_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix (Numeric Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Bivariate Analysis — Survival Rate by Feature
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Survival by Pclass
pclass_surv = train_df.groupby('Pclass')['Survived'].mean() * 100
sns.barplot(x=pclass_surv.index, y=pclass_surv.values, palette='Blues_d', ax=axes[0, 0])
axes[0, 0].set_title('Survival Rate by Pclass', fontsize=13, fontweight='bold')
axes[0, 0].set_ylabel('Survival Rate (%)')
for i, v in enumerate(pclass_surv.values):
    axes[0, 0].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11)

# Survival by Sex
sex_surv = train_df.groupby('Sex')['Survived'].mean() * 100
sns.barplot(x=sex_surv.index, y=sex_surv.values, palette=['#e74c3c', '#3498db'], ax=axes[0, 1])
axes[0, 1].set_title('Survival Rate by Sex', fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Survival Rate (%)')
for i, v in enumerate(sex_surv.values):
    axes[0, 1].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11)

# Survival by Embarked
emb_surv = train_df.groupby('Embarked')['Survived'].mean() * 100
sns.barplot(x=emb_surv.index, y=emb_surv.values, palette='Greens_d', ax=axes[0, 2])
axes[0, 2].set_title('Survival Rate by Embarked Port', fontsize=13, fontweight='bold')
axes[0, 2].set_ylabel('Survival Rate (%)')
for i, v in enumerate(emb_surv.values):
    axes[0, 2].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11)

# Survival by Age Band
train_df['AgeBand_temp'] = pd.cut(train_df['Age'].fillna(train_df['Age'].median()),
                              bins=[0, 12, 20, 40, 60, 100],
                              labels=['Child', 'Teen', 'Adult', 'Middle', 'Senior'])
age_surv = train_df.groupby('AgeBand_temp')['Survived'].mean() * 100
sns.barplot(x=age_surv.index, y=age_surv.values, palette='Oranges_d', ax=axes[1, 0])
axes[1, 0].set_title('Survival Rate by Age Group', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Survival Rate (%)')
for i, v in enumerate(age_surv.values):
    axes[1, 0].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11)

# Survival by Family Size (SibSp + Parch + 1)
train_df['FamilySize_temp'] = train_df['SibSp'] + train_df['Parch'] + 1
fam_surv = train_df.groupby('FamilySize_temp')['Survived'].mean() * 100
sns.barplot(x=fam_surv.index.astype(str), y=fam_surv.values, palette='Purples_d', ax=axes[1, 1])
axes[1, 1].set_title('Survival Rate by Family Size', fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel('Survival Rate (%)')
axes[1, 1].set_xlabel('Family Size')

# Pclass vs Sex (grouped bar)
pclass_sex = train_df.groupby(['Pclass', 'Sex'])['Survived'].mean().unstack() * 100
pclass_sex.plot(kind='bar', color=['#e74c3c', '#3498db'], ax=axes[1, 2])
axes[1, 2].set_title('Survival Rate: Pclass × Sex', fontsize=13, fontweight='bold')
axes[1, 2].set_ylabel('Survival Rate (%)')
axes[1, 2].legend(title='Sex')
axes[1, 2].set_xticklabels(axes[1, 2].get_xticklabels(), rotation=0)

# Clean up temp columns
train_df.drop(columns=['AgeBand_temp', 'FamilySize_temp'], inplace=True, errors='ignore')

plt.suptitle('Bivariate Analysis: Survival Rate by Feature', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🔧 Feature Engineering Walkthrough

Now we'll apply the feature engineering pipeline from our `src` modules.

In [ ]:
# Cell 12: Apply Feature Engineering Pipeline
from src.feature_engineering import FeatureEngineer, extract_title, normalize_title, extract_deck
from src.preprocessor import TitanicPreprocessor

# Initialize engineer with all features enabled
engineer = FeatureEngineer(
    create_family_features=True,
    create_title_feature=True,
    create_fare_bins=True,
    create_age_bands=True,
    create_deck_feature=True,
    create_interactions=True,
)

# Transform training data
train_engineered = engineer.transform(train_df.copy())

print(f"Original features:  {train_df.shape[1]}")
print(f"Engineered features: {train_engineered.shape[1]}")
print(f"\nNew features added:")
new_features = sorted(set(train_engineered.columns) - set(train_df.columns))
for f in new_features:
    print(f"  ➕ {f}")

In [ ]:
# Cell 13: Title Extraction Analysis
print("=" * 50)
print("TITLE EXTRACTION ANALYSIS")
print("=" * 50)

# Extract titles from names
titles = train_df['Name'].apply(extract_title).apply(normalize_title)
title_counts = titles.value_counts()
print(f"\nTitle distribution:\n{title_counts}")

# Survival rate by title
title_survival = pd.DataFrame({
    'Count': title_counts,
    'Survival Rate (%)': (train_df.groupby(titles)['Survived'].mean() * 100).round(2)
})
print(f"\n{title_survival}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(x=title_counts.index, y=title_counts.values, palette='viridis', ax=axes[0])
axes[0].set_title('Title Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

title_sr = train_df.groupby(titles)['Survived'].mean() * 100
sns.barplot(x=title_sr.index, y=title_sr.values, palette='RdYlGn', ax=axes[1])
axes[1].set_title('Survival Rate by Title', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].axhline(y=train_df['Survived'].mean()*100, color='red', linestyle='--', 
             label=f'Overall: {train_df["Survived"].mean()*100:.1f}%')
axes[1].legend()

plt.suptitle('Feature Engineering: Title Extraction', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: Family Size & IsAlone Analysis
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
train_df['IsAlone'] = (train_df['FamilySize'] == 1).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Family size distribution
fs_counts = train_df['FamilySize'].value_counts().sort_index()
sns.barplot(x=fs_counts.index.astype(str), y=fs_counts.values, palette='Blues', ax=axes[0])
axes[0].set_title('Family Size Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Family Size')
axes[0].set_ylabel('Count')

# Survival by family size
fs_surv = train_df.groupby('FamilySize')['Survived'].mean() * 100
sns.barplot(x=fs_surv.index.astype(str), y=fs_surv.values, palette='RdYlGn', ax=axes[1])
axes[1].set_title('Survival Rate by Family Size', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].axhline(y=38.4, color='red', linestyle='--', label='Overall ~38%')
axes[1].legend()

# IsAlone survival
alone_surv = train_df.groupby('IsAlone')['Survived'].mean() * 100
labels_alone = ['With Family', 'Alone']
colors_alone = ['#2ecc71', '#e74c3c']
axes[2].bar(labels_alone, alone_surv.values, color=colors_alone)
axes[2].set_title('Survival: Alone vs With Family', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Survival Rate (%)')
for i, v in enumerate(alone_surv.values):
    axes[2].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=12)

plt.suptitle('Feature Engineering: Family Features', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 15: Deck Letter Extraction from Cabin
decks = train_df['Cabin'].apply(extract_deck)
deck_counts = decks.value_counts().sort_values(ascending=False)
deck_surv = train_df.groupby(decks)['Survived'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x=deck_counts.index, y=deck_counts.values, palette='coolwarm', ax=axes[0])
axes[0].set_title('Deck Distribution (M = Missing)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(deck_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=10)

sns.barplot(x=deck_surv.index, y=deck_surv.values, palette='RdYlGn_r', ax=axes[1])
axes[1].set_title('Survival Rate by Deck', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].axhline(y=38.4, color='blue', linestyle='--', label='Overall ~38%')
axes[1].legend()

plt.suptitle('Feature Engineering: Deck Extraction from Cabin', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 16: Full Preprocessing Pipeline Demo
preprocessor = TitanicPreprocessor(
    fill_age_by_group=True,
    encode_categorical=True,
    scale_features=True,
    extract_deck=True,
)

# Fit on training data
train_processed = preprocessor.fit_transform(train_df.copy())

print(f"Processed shape: {train_processed.shape}")
print(f"\nFinal feature list for modeling:")
feature_names = preprocessor.get_feature_names(train_df)
exclude = {'PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin'}
model_features = [f for f in feature_names if f not in exclude]
for i, f in enumerate(model_features, 1):
    print(f"  {i:2d}. {f}")

print(f"\nTotal model-ready features: {len(model_features)}")
print(f"Remaining missing values: {train_processed[model_features].isnull().sum().sum()}")

In [ ]:
# Cell 17: Summary Statistics After Feature Engineering
print("=" * 70)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 70)
print(f"\nOriginal features:     {train_df.shape[1]}")
print(f"After engineering:      {train_processed.shape[1]}")
print(f"New features created:   {train_processed.shape[1] - train_df.shape[1]}")
print(f"Model-ready features:   {len(model_features)}")
print(f"\nFeature categories:")
print(f"  • Original numeric:  {len([c for c in model_features if c in train_df.select_dtypes(include=[np.number])])}")
print(f"  • Engineered:         {len(new_features)}")
print(f"  • Encoded categoricals: {len([c for c in model_features if '_encoded' in c])}")
print(f"\n✅ EDA & Feature Engineering complete!")
print(f"👉 Next step: Run 02_modeling_and_submission.ipynb")